In [1]:
import os
import sys
from typing import Dict, Any

import json
import jsonlines
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

sys.path.append(f"{os.environ['MDS']}/code")
from project_utils import load_config

pd.set_option('display.precision', 2)

/home/adeluci2/.conda/envs/mds/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(f"{os.environ['MDS']}/code/experiments.json") as f:
    experiments_config = json.load(f)

In [3]:
# Most of the models are using the BART tokenizer 
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large")

/home/adeluci2/.conda/envs/mds/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


# Define Metrics

In [4]:
def collect_stats(row: Dict[str, Any]) -> Dict[str, Any]:
    stats = {}
    # 1. # of source documents
    stats["docs"] = len(row["docs"])
    stats["summary_length"] = len(row["target"].split())
    stats["doc_length_raw"] = [len(tokenizer.encode(d))-2 for d in row["docs"]]  # Subtract 2 to remove BOS/EOS
    stats["doc_length"] = np.mean(stats["doc_length_raw"])
    stats["id"] = row["id"]
    return stats


# Process Data

In [5]:
all_stats = []
for dataset in experiments_config["datasets"]:
    for split in ["train", "val", "test"]:
        with jsonlines.open(f"{os.environ['MDS']}/data/{dataset}/{split}.jsonl") as reader:
            for entry in reader.iter():
                temp = collect_stats(entry)
                temp["dataset"] = dataset
                temp["split"] = split
                all_stats.append(temp)

df = pd.DataFrame(all_stats)

Token indices sequence length is longer than the specified maximum sequence length for this model (1050 > 1024). Running this sequence through the model will result in indexing errors


In [6]:
summary = df.groupby(["dataset", "split"]).agg({
    "docs": ["mean", "std"],
    "summary_length": ["mean", "std"],
    "doc_length": ["mean", "std"],
    "id": "count"
})
summary

docs       summary_length        doc_length           \
                         mean   std           mean    std       mean      std   
dataset          split                                                          
convosumm/reddit test   14.42  5.55          57.78  12.04      53.02    26.60   
                 train  14.98  6.28          58.32  12.95      49.46    26.38   
                 val    16.74  7.21          59.18  14.99      52.88    27.29   
multinews        test    2.73  1.08         217.00  67.20     982.98  1779.55   
                 train   2.76  1.12         218.27  68.04     982.95  1851.20   
                 val     2.70  1.04         216.72  67.67     980.06  1163.94   
multinews_plus   test    2.53  0.88         230.94  69.57     226.07    64.15   
                 train   2.55  0.91         233.54  69.98     224.30    64.96   
                 val     2.51  0.86         231.45  69.64     226.59    63.35   
multixscience    test    3.55  1.95         106.35  41.60     195.32    48.01   
                 train   3.58  2.05         106.82  42.14     197.03    47.91   
                 val     3.55  1.99         104.03  41.38     196.98    48.84   

                           id  
                        count  
dataset          split         
convosumm/reddit test     250  
                 train    201  
                 val       50  
multinews        test    5621  
                 train  44964  
                 val     5621  
multinews_plus   test    4648  
                 train  37057  
                 val     4576  
multixscience    test    1901  
                 train  11523  
                 val     1937

In [7]:
print(summary.to_csv())

,,docs,docs,summary_length,summary_length,doc_length,doc_length,id
,,mean,std,mean,std,mean,std,count
dataset,split,,,,,,,
convosumm/reddit,test,14.416,5.552500793357977,57.78,12.041244381917519,53.01828655946646,26.602476950106244,250
convosumm/reddit,train,14.975124378109452,6.282067980327235,58.318407960199,12.945582623147411,49.458091704914686,26.380567477628542,201
convosumm/reddit,val,16.74,7.213394570137544,59.18,14.994815430538592,52.87596071670372,27.289620505029006,50
multinews,test,2.7262053015477674,1.0825064791569272,217.00071161714996,67.19843138785448,982.9849397666912,1779.5469909274655,5621
multinews,train,2.758406725380304,1.1248445565996832,218.2742193755004,68.04126012273362,982.9478492582426,1851.2032678137657,44964
multinews,val,2.700053371286248,1.0399741633791657,216.72033446006049,67.67051793409145,980.059950568023,1163.936428827731,5621
multinews_plus,test,2.5316265060240966,0.8759207278634588,230.93911359724612,69.56505992008144,226.0669583948037,64.148816917

In [8]:
for row_id, row in summary.iterrows():
    print(f"""{row_id[0]} & {row_id[1]} & {row["id"]["count"]} & {row["docs"]["mean"]:.2f} ({row["docs"]["std"]:.2f}) & {row["summary_length"]["mean"]:.2f} ({row["summary_length"]["std"]:.2f}) & {row["doc_length"]["mean"]:.2f} ({row["doc_length"]["std"]:.2f}) \\\\""")

convosumm/reddit & test & 250.0 & 14.42 (5.55) & 57.78 (12.04) & 53.02 (26.60) \\
convosumm/reddit & train & 201.0 & 14.98 (6.28) & 58.32 (12.95) & 49.46 (26.38) \\
convosumm/reddit & val & 50.0 & 16.74 (7.21) & 59.18 (14.99) & 52.88 (27.29) \\
multinews & test & 5621.0 & 2.73 (1.08) & 217.00 (67.20) & 982.98 (1779.55) \\
multinews & train & 44964.0 & 2.76 (1.12) & 218.27 (68.04) & 982.95 (1851.20) \\
multinews & val & 5621.0 & 2.70 (1.04) & 216.72 (67.67) & 980.06 (1163.94) \\
multinews_plus & test & 4648.0 & 2.53 (0.88) & 230.94 (69.57) & 226.07 (64.15) \\
multinews_plus & train & 37057.0 & 2.55 (0.91) & 233.54 (69.98) & 224.30 (64.96) \\
multinews_plus & val & 4576.0 & 2.51 (0.86) & 231.45 (69.64) & 226.59 (63.35) \\
multixscience & test & 1901.0 & 3.55 (1.95) & 106.35 (41.60) & 195.32 (48.01) \\
multixscience & train & 11523.0 & 3.58 (2.05) & 106.82 (42.14) & 197.03 (47.91) \\
multixscience & val & 1937.0 & 3.55 (1.99) & 104.03 (41.38) & 196.98 (48.84) \\


# Truncation

In [9]:
truncation_stats_path = "/home/adeluci2/scratch4-mdredze1/aadelucia/multidoc_summarization/results/experiments/truncation_stats.csv"

trunc_df = pd.read_csv(truncation_stats_path)


In [10]:
trunc_df.groupby(["split", "model", "dataset"]).agg({
    "truncated_tokens": ["mean", "std"],
    "truncated": "sum"
})

truncated_tokens          truncated
                                             mean      std       sum
split model     dataset                                             
test  ConvoSumm convosumm/reddit             0.00     0.00         0
                multinews                  289.27  2272.24       699
                multixscience                6.10    69.47        25
      PRIMERA   convosumm/reddit             0.00     0.00         0
                multinews                  695.29  6909.78       835
                multixscience                0.00     0.00         0
val   ConvoSumm convosumm/reddit             0.00     0.00         0
                multinews                  282.81  2339.89       720
                multixscience                7.51    80.74        36
      PRIMERA   convosumm/reddit             0.00     0.00         0
                multinews                  579.10  3345.25       833
                multixscience                0.00     0.00         0